# Canonicalization-via-PPO  ---  Colab runbook

End-to-end recipe for running the project on a free Colab T4 (or Pro A100):

1. Confirm GPU.
2. Mount Google Drive (so checkpoints survive disconnects).
3. Get the project code onto the Colab VM (3 options).
4. Symlink `checkpoints/` and `logs/` to Drive.
5. Install deps + sanity-check.
6. Train.
7. Monitor with TensorBoard.
8. Resume after a disconnect.
9. Test / download checkpoints.

**Why Drive?** Colab's `/content/` filesystem is wiped when the VM dies. Drive persists, and the trainer already saves a checkpoint every 25 PPO updates. Symlinking means the training script doesn't even know it's writing to Drive.

## 1. Verify GPU

If this prints CPU or 'No CUDA': **Runtime → Change runtime type → T4 GPU** (or A100 if Pro), then re-run.

In [ ]:
import subprocess, sys
print('python:', sys.version.split()[0])
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode()
    print('GPU   :', out.strip())
except Exception as e:
    print('GPU   : NONE  (', e, ')')

## 2. Mount Google Drive

Creates `/content/drive/MyDrive/canon_ppo/` to hold checkpoints, logs, and (optionally) the dataset. You only do this once per session; it survives across reconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/canon_ppo'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/logs',        exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data',        exist_ok=True)
print('Drive root:', DRIVE_ROOT)
!ls -la /content/drive/MyDrive/canon_ppo

## 3. Get the code onto Colab  ---  pick ONE option

I recommend Option A (GitHub) if your repo is reachable, otherwise B (zip).
Option C is good once the project is already in your Drive (e.g. you ran 3.A once and copied).

### Option A — GitHub clone (recommended)

Edit `REPO_URL` to your repo. Public repos work as-is; private repos need a personal access token in the URL: `https://<TOKEN>@github.com/<you>/<repo>.git`.

In [ ]:
REPO_URL = 'https://github.com/<your-username>/<your-repo>.git'
PROJECT_DIR = '/content/canon_ppo'

%cd /content
![ -d {PROJECT_DIR} ] && rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!ls

### Option B — upload a zip from your laptop

On your laptop:
```
cd "Spring 2026/CS 58700"
zip -r Final_Project_2.zip Final_Project_2 -x '*/checkpoints/*' '*/logs/*' '*/data/images/*' '*/__pycache__/*'
```

Then run this cell, click **Choose files**, and pick the zip.

In [ ]:
from google.colab import files
uploaded = files.upload()
import os
zip_name = next(iter(uploaded))
PROJECT_DIR = '/content/canon_ppo'
!rm -rf {PROJECT_DIR}
!unzip -q {zip_name} -d /content/_extracted
inner = !ls /content/_extracted
src = '/content/_extracted/' + inner[0]
!mv {src} {PROJECT_DIR}
!rm -rf /content/_extracted {zip_name}
%cd {PROJECT_DIR}
!ls

### Option C — copy from Drive (after first-time upload)

Once the project lives at `/content/drive/MyDrive/canon_ppo_code/`, this is the fastest reconnect path.

In [ ]:
PROJECT_DIR = '/content/canon_ppo'
!rm -rf {PROJECT_DIR}
!cp -r /content/drive/MyDrive/canon_ppo_code {PROJECT_DIR}
%cd {PROJECT_DIR}
!ls

## 4. Symlink `checkpoints/` and `logs/` to Drive

After this, every `torch.save(...)` call inside `src/ppo.py` lands in your Drive automatically. No code changes needed.

Also symlink `data/images` so the 100-image download is cached across sessions.

In [ ]:
import os, pathlib
PROJECT_DIR = '/content/canon_ppo'
DRIVE_ROOT = '/content/drive/MyDrive/canon_ppo'

os.chdir(PROJECT_DIR)
for src_name in ('checkpoints', 'logs'):
    target = f'{DRIVE_ROOT}/{src_name}'
    link = f'{PROJECT_DIR}/{src_name}'
    !rm -rf {link}
    !ln -s {target} {link}

os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
!rm -rf {PROJECT_DIR}/data/images
!ln -s {DRIVE_ROOT}/data {PROJECT_DIR}/data/images

!ls -la {PROJECT_DIR} | grep -E '(checkpoints|logs|data)'

## 5. Install dependencies + sanity check

Two quick gates before doing anything heavy:

- `quick_check.py` runs the rotation + env + a tiny DINOv2-small forward pass (no VLM, ~30 s).
- `train.py --config configs/debug.yaml` runs 30 PPO updates with the synthetic reward (~1 min). If `final_abs_angle_mean` drops below ~10° here, the PPO loop itself is healthy.

In [ ]:
%cd /content/canon_ppo
!pip install -q -r requirements.txt
!python -c "import torch; print('torch', torch.__version__, 'cuda?', torch.cuda.is_available())"

In [ ]:
!python scripts/quick_check.py

In [ ]:
!python scripts/train.py --config configs/debug.yaml

## 6. Download the 100-image dataset

Saved into `data/images/` which is symlinked to Drive — it persists, so future sessions skip the download.

In [ ]:
!python scripts/download_data.py --config configs/colab_500m.yaml

## 7. Train  ---  the real run

Uses `configs/colab_500m.yaml`:
- ViT-Huge (~632M) policy backbone, frozen.
- Qwen2-VL-2B fp16 reward model.
- 200 PPO updates with checkpoint every 25 → 8 checkpoints in Drive over the run.

Expected wall-clock on a free T4: **~60-90 minutes**.

If you get a HuggingFace gated-model warning when downloading Qwen, run
`from huggingface_hub import login; login()` once with your HF token.

In [ ]:
!python scripts/train.py --config configs/colab_500m.yaml

## 8. Live monitoring (optional, run in parallel)

Open a second cell while the training above is still running. Inline TensorBoard reads the symlinked log dir straight from Drive.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/canon_ppo/logs

## 9. If Colab disconnects: resume from the latest checkpoint

After a disconnect: re-run cells 1, 2, 3, 4, 5 (just `pip install`, not the sanity scripts), then this cell. It auto-finds the newest `policy_update*.pt` in Drive and passes `--resume`.

In [ ]:
import glob, os, re
ckpt_glob = '/content/canon_ppo/checkpoints/canon_ppo_colab_500m/policy_update*.pt'
ckpts = sorted(glob.glob(ckpt_glob),
               key=lambda p: int(re.search(r'update(\d+)', p).group(1)))
if not ckpts:
    print('No checkpoint found; starting from scratch.')
    !python scripts/train.py --config configs/colab_500m.yaml
else:
    last = ckpts[-1]
    print('Resuming from', last)
    !python scripts/train.py --config configs/colab_500m.yaml --resume {last}

## 10. Test-time canonicalization

Run the trained policy on the same image pool with the convergence-based stopping rule (|reward delta| < tolerance for `patience` consecutive steps).

In [ ]:
!python scripts/test.py \
    --config configs/colab_500m.yaml \
    --checkpoint /content/canon_ppo/checkpoints/canon_ppo_colab_500m/policy_final.pt \
    --num_images 32 \
    --save_trace /content/canon_ppo/logs/canon_ppo_colab_500m/test_trace.npz

In [ ]:
!python scripts/test.py \
    --config configs/colab_500m.yaml \
    --checkpoint /content/canon_ppo/checkpoints/canon_ppo_colab_500m/policy_final.pt \
    --image_path /content/canon_ppo/data/images/img_0000.jpg \
    --initial_angle 73 \
    --save_trace /content/canon_ppo/logs/canon_ppo_colab_500m/single_trace.npz

## 11. Download a checkpoint to your laptop

Drive already has them, but if you want a direct browser download:

In [ ]:
from google.colab import files
files.download('/content/canon_ppo/checkpoints/canon_ppo_colab_500m/policy_final.pt')

## Notes

- Free Colab GPU sessions are roughly 4 hours; 200 updates fit comfortably. If you only get T4 idle disconnects (90 min), checkpoints every 25 updates means losing at most ~10 min of progress.
- **Slim checkpoints are on by default** whenever the backbone is frozen: only the trainable actor/critic heads + optimizer state are saved (~5-10 MB each). The frozen backbone is reloaded from HuggingFace at resume time. So 8 checkpoints ≈ 80 MB of Drive, not 20 GB.
- **Do not** unfreeze the backbone on a free T4. Set `policy.freeze_backbone: false` only on Pro A100 (~22 GB needed). When unfrozen, the trainer automatically saves full checkpoints (~2.5 GB each).